<style>
.jp-RenderedHTMLCommon { font-size: 22px !important; line-height: 1.8 !important; }
.jp-RenderedHTMLCommon h1 { font-size: 2.2em !important; }
.jp-RenderedHTMLCommon h2 {
  font-size: 1.7em !important;
  border-left: 5px solid #ff9800 !important;
  padding: 10px 0 10px 18px !important;
  margin-top: 40px !important;
  margin-bottom: 16px !important;
  background: linear-gradient(90deg, rgba(33,150,243,0.08) 0%, transparent 60%) !important;
  border-radius: 4px !important;
}
.jp-RenderedHTMLCommon h3 {
  font-size: 1.35em !important;
  border-bottom: 2px solid rgba(255,152,0,0.3) !important;
  padding-bottom: 8px !important;
  margin-top: 30px !important;
  margin-bottom: 12px !important;
  color: #64b5f6 !important;
}
.jp-RenderedHTMLCommon h4 {
  font-size: 1.15em !important;
  font-style: italic !important;
  color: #90caf9 !important;
  margin-top: 22px !important;
}
</style>

<div style="background: linear-gradient(135deg, #2e1f0a 0%, #5d4037 50%, #e65100 100%); padding: 40px 45px; border-radius: 16px; margin-bottom: 30px; border: 1px solid rgba(255,204,128,0.15); max-width: 95%; overflow: hidden; box-sizing: border-box;">
  <div style="font-size: 15px; font-weight: 600; letter-spacing: 3px; text-transform: uppercase; color: #ffcc80; margin-bottom: 8px;">Chapter 05</div>
  <div style="font-size: 36px; font-weight: 800; color: #ffffff; margin-bottom: 16px; line-height: 1.2;">Harmful Content Detection</div>
  <div style="width: 60px; height: 4px; background: linear-gradient(90deg, #ff9800, #ffcc80); border-radius: 2px; margin-bottom: 18px;"></div>
  <div style="font-size: 17px; color: #b0bec5; line-height: 1.6;">Many social media platforms such as Facebook, LinkedIn, and Twitter have standard guidelines to enforce integrity and make their platforms safe for users. In this chapter, we design a system that proactively monitors new posts, detects harmful content, and removes or demotes them.</div>
  <div style="margin-top: 20px; font-size: 13px; color: #546e7a;">Source: ByteByteGo — Machine Learning System Design Interview</div>
</div>

Many social media platforms such as Facebook [1], LinkedIn [2], and Twitter [3] have standard guidelines to enforce integrity and make their platforms safe for users. These guidelines prohibit certain user behaviors, activities, and content that are harmful to the community. It is essential to have technologies and resources in place to identify harmful content and bad actors. We can divide the focus of integrity enforcement into two categories:

* Harmful content: Posts that contain violence, nudity, self-harm, hate speech, etc.
* Bad acts/bad actors: Fake accounts, spam, phishing, organized unethical activities, and other unsafe behaviors.

In this chapter, we focus on detecting posts that might contain harmful content. In particular, we design a system that proactively monitors new posts, detects harmful content, and removes or demotes them if the content violates the platform's guidelines. To understand how companies build a harmful content detection system in practice, refer to [4] [5] [6].

![Figure 5.1: Harmful content detection system](assets/ch5-01-1-ROVCWIXC.png)

<div style="border-left: 5px solid #f57c00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,152,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Clarifying Requirements</span>
</div>

Here is a typical interaction between a candidate and an interviewer.
<div style="border: 1px solid rgba(255,173,50,0.3); border-radius: 8px; margin: 28px 0; overflow: hidden; font-family: system-ui, -apple-system, sans-serif;">
  <div style="background: rgba(255,173,50,0.08); padding: 10px 20px; border-bottom: 1px solid rgba(255,173,50,0.3); display: flex; align-items: center; gap: 8px;">
    <span style="font-size: 1.1em;">&#128483;</span>
    <span style="font-size: 0.85em; font-weight: 700; color: #ffad32; text-transform: uppercase; letter-spacing: 1.5px;">Interview Transcript</span>
  </div>
  <div style="padding: 24px 20px 24px 28px; position: relative;">
    <div style="position: absolute; left: 38px; top: 24px; bottom: 24px; width: 2px; background: rgba(255,173,50,0.3);"></div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Does the system detect both harmful content and bad actors?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Both are equally important. For simplicity, let's focus on detecting harmful content only.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Should a post only contain text, or are images and videos allowed?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">The content of a post can be text, image, video, or any combination of these.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">What languages are supported? Is it English only?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">The system should detect harmful content in various languages. For simplicity, assume we can use a pre-trained multilingual model to embed the textual content.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Which specific categories of harmful content are we looking to identify? I can think of violence, nudity, hate speech, misinformation, etc. Are there other harmful categories to consider?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Great, you brought up the major ones. Misinformation is more complex and controversial. For simplicity, let's not focus on misinformation.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Are there any human annotators available to label posts manually?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">The platform receives more than 500 million posts each day. Asking humans to label all of them would be very expensive and time-consuming. However, you can assume human annotation is available to label a limited number of posts, say 10,000 per day.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Allowing users to report harmful content is beneficial for understanding where the system is failing. Can I assume the system has that feature?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Good point. Yes, users can report harmful posts.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Should we explain why a post is deemed harmful and removed?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Yes. Explaining to users why we remove a post is essential. It helps users to ensure they align their future posts with the guidelines.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">What is the system's latency requirement? Do we need a real-time prediction, i.e., the system detects harmful content immediately and blocks it, or can we rely on batch prediction, i.e., detecting harmful content offline hourly or daily?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">This is a very important question. What are your thoughts?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffad32; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffad32; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffad32; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffad32; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">In my opinion, the requirements for different harmful content might vary. For example, violent content may require real-time solutions, while for others, late detection may work.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Those are fair assumptions.</div>
      </div>
    </div>

  </div>
</div>
So, let's summarize the problem statement. We will design a harmful content detection system, which identifies harmful posts, then deletes or demotes them and informs the user why the post was identified as harmful. A post's content can be text, image, video, or any combination of these, and the content can be in different languages. Users can report harmful posts.

<div style="border-left: 5px solid #f57c00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,152,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Frame the Problem as an ML Task</span>
</div>

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Defining the ML objective</span>
</div>

We define our ML objective as accurately predicting harmful posts. The reason is that if we can accurately detect harmful posts, we can remove or demote them, leading to a safer platform.

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Specifying the system's input and output</span>
</div>

The system receives a post as input, and outputs the probability that the post is harmful.

![Figure 5.2: A harmful content detection system's input-output](assets/ch5-02-1-BJSVLTF6.png)

Let's dive deeper into the input post. As shown in Figure 5.3, a post can be heterogeneous and potentially multimodal.

![Figure 5.3: Heterogeneous post data](assets/ch5-03-1-4W7DMAXE.png)

To make accurate predictions, the system should consider all modalities. Let's discuss two commonly used fusing methods to combine heterogeneous data: late fusion and early fusion.

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Late fusion</span>
</div>

With late fusion, ML models process different modalities independently, then combine their predictions to make a final prediction. The figure below illustrates how late fusion works.

![Figure 5.4: Late fusion](assets/ch5-04-1-ZXXZUOPW.png)

The advantage of late fusion is that we can train, evaluate, and improve each model independently.

However, late fusion has two major disadvantages. First, to train these individual models, we need to have separate training data for each modality, which can be time-consuming and expensive.

Second, the combination of the modalities might be harmful, even if each is benign in isolation. This is frequently the case with memes that combine images and text. In these cases, late fusion fails to predict whether the content is harmful. This is because each modality is benign, so the models predict benignity when processing each modality. The fusion layer output is benign because the output of each separate modality is benign. But this is incorrect, as the combination of modalities can be harmful.

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Early fusion</span>
</div>

With early fusion, the modalities are combined first, and then the model makes a prediction. Figure 5.5 illustrates how early fusion works.

![Figure 5.5: Early fusion](assets/ch5-05-1-E6HNOFG4.png)

Early fusion has two major advantages. First, it is unnecessary to collect training data separately for each modality. Since there is a single model to train, we only need to collect training data for that model. Second, the model considers all the modalities, so if each modality is benign, but their combination is harmful, then the model can potentially capture this in the unified feature vector.

However, learning this task is more difficult for the model due to the complex relationships between modalities. In the absence of sufficient training data, it is challenging for the model to learn complex relationships and make good predictions.

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Which fusion method should we use?</span>
</div>

The early fusion method is used because it allows us to capture posts that may be harmful overall, even if each modality is benign on its own. Additionally, with around 500 million posts being published every day, the model has enough data to learn the task.

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Choosing the right ML category</span>
</div>

In this section, we examine the following ML category options:

* Single binary classifier
* One binary classifier per harmful class
* Multi-label classifier
* Multi-task classifier

<strong style="color: #ffad32;">Single binary classifier</strong>

In this option, a model takes the fused features as input and predicts the probability of the post being harmful (Figure 5.6). Since the output is a binary outcome, the model is a binary classifier.

![Figure 5.6: A single binary classifier](assets/ch5-06-1-GAYWQ7X3.png)

The shortcoming of this option is that it's difficult to determine which class of harm, such as violence, a post belongs to. This limitation causes two main issues:

* It is not easy to inform users why we take down a post as the system only outputs a binary value indicating whether the post as a whole is harmful or not. We have no clue which particular class of harm the post belongs to.
* It is not easy to identify harmful classes in which the system doesn't perform well, meaning we cannot improve the system for underperforming classes.

Since it is essential to explain why a post is removed, a single binary classifier is not a good option.

<strong style="color: #ffad32;">One binary classifier per harmful class</strong>

In this option, we adopt one binary classifier for each harmful class. As Figure 5.7 shows, each model determines if a post belongs to a specific harmful class or not. Each model takes fused features as input and predicts the probability of the post being classified as a harmful class.

![Figure 5.7: One binary classifier per harmful class](assets/ch5-07-1-IUZCP5VM.png)

The advantage of this option is that we can explain to users why a post was taken down. In addition, we can monitor different models and improve them independently.

However, this option has one major drawback. Since we have multiple models, they must be trained and maintained separately. Training these models separately is time-consuming and expensive.

<strong style="color: #ffad32;">A multi-label classifier</strong>

In multi-label classification, the data point we want to classify may belong to an arbitrary number of classes. In this option, a single model is used as a multi-label classifier. As Figure 5.8 shows, the input to the model is the fused features, and the model predicts probabilities for each harmful class.

![Figure 5.8: A multi-label classifier](assets/ch5-08-1-S3IY7HXM.png)

By using a shared model for all harmful classes, training and maintaining the model is less costly. If you would like to learn more about this method, refer to the approach called WPIE [7].

However, predicting the probabilities of each harmful class using a shared model isn't ideal, as the input features may need to be transformed differently.

<strong style="color: #ffad32;">Multi-task classifier</strong>

Multi-task learning refers to the process in which a model learns multiple tasks simultaneously. This allows the model to learn similarities between tasks. By doing so, we avoid unnecessary computations when a certain input transformation is beneficial for multiple tasks.

In our case, we treat different classes of harm, such as violence and nudity, as different tasks and use a multi-task classification model to learn each task. As Figure 5.9 shows, multi-task classification has two stages: shared layers and task-specific layers.

![Figure 5.9: Multi-task classification overview](assets/ch5-09-1-GWHRFHAQ.png)

<strong style="color: #ffad32;">Shared layers.</strong> A shared layer, as shown in Figure 5.10, is a set of hidden layers that transform input features into new ones. These newly transformed features are used to make predictions for each of the harmful classes.

![Figure 5.10: Shared layers](assets/ch5-10-1-ZKO4GCMD.png)

<strong style="color: #ffad32;">Task-specific layers.</strong> Task-specific layers are a set of independent ML layers (also called classification heads). Each classification head transforms features in a way that is optimal for predicting a specific harm probability.

![Figure 5.11: Task-specific layers](assets/ch5-11-1-UFN5H4Y5.png)

Multi-task classification has three advantages. First, it is not expensive to train or maintain since we use a single model. Second, the shared layers transform the features in a way that is beneficial for each task. This prevents redundant computations and makes multi-task classification efficient. Lastly, the training data for each task contributes to the learning of other tasks. This is especially helpful when limited data is available for a particular task.

Because of these advantages, we employ a multi-task classification method. Figure 5.12 shows how we frame the problem.

![Figure 5.12: Frame the problem as an ML task](assets/ch5-12-1-GCKNMDGB.png)

In [ ]:
# ── Code Augmentation: Multi-Task Classification Architecture ──

import torch
import torch.nn as nn

class MultiTaskHarmClassifier(nn.Module):
    """
    A multi-task classification model for harmful content detection.
    Shared layers learn common representations, while task-specific heads
    predict probabilities for each harmful class independently.
    """
    def __init__(self, input_dim=768, shared_dim=256, num_tasks=4):
        super().__init__()
        
        # Shared layers (common feature transformation)
        self.shared_layers = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, shared_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        
        # Task-specific classification heads
        # Each head is a binary classifier for one harm category
        self.task_heads = nn.ModuleDict({
            'violence':   nn.Sequential(nn.Linear(shared_dim, 64), nn.ReLU(), nn.Linear(64, 1)),
            'nudity':     nn.Sequential(nn.Linear(shared_dim, 64), nn.ReLU(), nn.Linear(64, 1)),
            'hate_speech': nn.Sequential(nn.Linear(shared_dim, 64), nn.ReLU(), nn.Linear(64, 1)),
            'self_harm':  nn.Sequential(nn.Linear(shared_dim, 64), nn.ReLU(), nn.Linear(64, 1)),
        })
        
    def forward(self, fused_features):
        # Pass through shared layers
        shared_repr = self.shared_layers(fused_features)
        
        # Pass through each task-specific head
        outputs = {}
        for task_name, head in self.task_heads.items():
            outputs[task_name] = torch.sigmoid(head(shared_repr))
        
        return outputs

# Example: Simulate a fused feature vector for a batch of 4 posts
model = MultiTaskHarmClassifier(input_dim=768, shared_dim=256)
dummy_fused_features = torch.randn(4, 768)

with torch.no_grad():
    predictions = model(dummy_fused_features)

print("Predictions for a batch of 4 posts:")
for task, probs in predictions.items():
    print(f"  {task:12s}: {probs.squeeze().tolist()}")

<div style="border-left: 5px solid #f57c00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,152,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Data Preparation</span>
</div>

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Data engineering</span>
</div>

We have the following data available:

* Users
* Posts
* User-post interactions

<strong style="color: #ffad32;">Users</strong>

A user data schema is shown below.

| ID | Username | Age | Gender | City | Country | Email |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |

*Table 5.1: User data schema*

<strong style="color: #ffad32;">Posts</strong>

Post data contain fields such as author, time of upload, etc. Table 5.2 shows some of the most important attributes. In practice, we typically have hundreds of attributes associated with each post.

| Post ID | Author ID | On-device | Timestamp | Textual content | Images or videos | Links |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 1 | 73.93.220.240 | 1658469431 | Today, I am starting my diet. | http://cdn.mysite.com/u1.jpg | - |
| 2 | 11 | 89.42.110.250 | 1658471428 | The video amazed me! Please donate | http://cdn.mysite.com/t3.mp4 | gofundme.com/f/3u1njd32 |
| 3 | 4 | 39.55.180.020 | 1658489233 | What is a good restaurant in the Bay area? | http://cdn.mysite.com/t5.jpg | - |

*Table 5.2: Post data*

<strong style="color: #ffad32;">User-post interactions</strong>

User-post interaction data primarily contain users' reactions to posts, such as likes, comments, saves, shares, etc. Users can also report a post as harmful or request an appeal. Table 5.3 shows what the data might look like.

| User ID | Post ID | Interaction type | Interaction value | Timestamp |
| :--- | :--- | :--- | :--- | :--- |
| 11 | 6 | Impression | - | 1658450539 |
| 4 | 20 | Like | - | 1658451341 |
| 11 | 7 | Comment | This is disgusting | 1658451365 |
| 4 | 20 | Share | - | 1658435948 |
| 11 | 7 | Report | violence | 1658451849 |

*Table 5.3: User-post interaction data*

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Feature engineering</span>
</div>

In the "Frame the problem as an ML task" section, we framed the problem as a multi-task classification task where the input is the post. In this section, we explore predictive features that can be derived from a post. A post might comprise the following elements:

* Textual content
* Image or video
* User reactions to the post
* Author
* Contextual information

Let's look at each element.

<strong style="color: #ffad32;">Textual content</strong>

The textual content of a post can be used to determine whether a post is harmful or not. As described in Chapter 4 YouTube Video Search, text data is usually prepared in two steps:

* Text preprocessing (e.g., normalization, tokenization)
* Vectorization: convert the preprocessed text into a meaningful feature vector

Let's focus on vectorization since it is unique to this chapter. To vectorize the text and extract a feature vector, statistical or ML-based methods can be used. Statistical methods such as BoW or TF-IDF are easy to implement and fast to compute. However, they cannot encode the semantics of the text. For our system, understanding the semantics of the textual content is important for determining harm, so we adopt the ML-based method. To convert text into a feature vector, we use a pre-trained Transformer-based language model such as BERT [8]. However, the original BERT has two issues:

* Producing the text embedding takes a long time due to the large size of the model. Because this is a slow process, using it for online predictions is not ideal.
* BERT was trained on English-only data. Thus, it does not produce meaningful embeddings for texts in other languages.

DistilmBERT [9], a more efficient variant of BERT, addresses those two issues. If two sentences have the same meaning but are in two different languages, their embeddings are very similar. If you are interested to learn more about multilingual language models, refer to [10].

<strong style="color: #ffad32;">Image or video</strong>

You can usually find out what a post is about by looking at the image or video within it. The following two steps are commonly used to prepare unstructured data such as images or videos.

* Preprocessing: decode, resize, and normalize the data.
* Feature extraction: after preprocessing, we use a pre-trained model to convert unstructured data to a feature vector. This allows us to represent an image or video by a feature vector. For images, a pre-trained image model such as CLIP's visual encoder [11] or SimCLR [12] are viable options. For videos, pre-trained models like VideoMoCo [13] might work well.

<strong style="color: #ffad32;">User reactions to the post</strong>

It is also possible to determine whether a post is harmful based on user reactions, especially when the content is ambiguous. As shown in Figure 5.13, with more comments, it becomes increasingly evident the post contains content related to self-harm.

![Figure 5.13: Post with self-harm concern](assets/ch5-13-1-LWFDENPP.png)

Since user reactions are crucial in determining harmful content, let's examine some of the features we can engineer based on them.

The number of likes, shares, comments, and reports: We usually scale these numerical values to speed up convergence during model training.

Comments: As demonstrated in Figure 5.13, comments can help us identify harmful content. For feature preparation, we convert comments into numerical representations by doing the following:

* Use the same pre-trained model we employed earlier to obtain the embedding of each comment.
* Aggregate (e.g., average) the embeddings to obtain a final embedding.

A summary of the features we have described so far can be found in Figure 5.14.

![Figure 5.14: Feature engineering for reactions and content](assets/ch5-14-1-JEVGAY27.png)

<strong style="color: #ffad32;">Author features</strong>

The author's past interactions can be used to determine if the post is harmful or not. Let's engineer features related to the post author.

<strong style="color: #ffad32;">Author's violation history</strong>

* Number of violations: This is a numerical value representing the number of times the author violated the guidelines in the past.
* Total user reports: A numerical value representing the number of times users reported the author's posts.
* Profane words rate: This is a numerical value representing the rate of profane words used in the author's previous posts and comments. A predefined list of profane words is used to determine whether a word is profane.

<strong style="color: #ffad32;">Author's demographics</strong>

* Age: A user's age is one of the most important predictive features.
* Gender: This categorical feature represents the user's gender. We use one-hot encoding to represent gender.
* City and country: Both the city and country take many distinct values. To represent the features, we use an embedding layer to convert city and country into feature vectors. Note that one-hot encoding is not an efficient method to represent the city and country because their representations would be long and sparse.

<strong style="color: #ffad32;">Account information</strong>

* Number of followers and followings
* Account age: This is a numerical value representing the age of the author's account. This is a predictive feature as accounts with a lower age are more likely to be spam or to violate integrity.

<strong style="color: #ffad32;">Contextual information</strong>

* Time of day: This is the time of day when the author made a post. We bucketize this into multiple categories, such as morning, noon, afternoon, evening or night. We use one-hot encoding to represent this feature.
* Device: The device the author uses, such as a smartphone or desktop computer. One-hot encoding is used to represent this feature.

Figure 5.15 summarizes some of the most important features of the harmful content detection system.

![Figure 5.15: Summary of feature engineering](assets/ch5-15-1-RGPR34CR.png)

In [ ]:
# ── Code Augmentation: Multimodal Feature Fusion Pipeline ──

import torch
import torch.nn as nn

class EarlyFusionEncoder(nn.Module):
    """
    Simulates early fusion of multimodal post data.
    Text, image, and metadata features are concatenated into a single vector
    and passed through a shared transformation layer.
    """
    def __init__(self, text_dim=768, image_dim=512, meta_dim=32, fused_dim=768):
        super().__init__()
        total_input_dim = text_dim + image_dim + meta_dim
        self.fusion_layer = nn.Sequential(
            nn.Linear(total_input_dim, fused_dim),
            nn.LayerNorm(fused_dim),
            nn.ReLU(),
        )
        
    def forward(self, text_emb, image_emb, meta_features):
        # Concatenate all modalities along the feature dimension
        concatenated = torch.cat([text_emb, image_emb, meta_features], dim=1)
        fused = self.fusion_layer(concatenated)
        return fused

# Simulate feature vectors for a batch of 4 posts
batch_size = 4
text_embeddings = torch.randn(batch_size, 768)   # e.g., from DistilmBERT
image_embeddings = torch.randn(batch_size, 512)  # e.g., from CLIP ViT
metadata_features = torch.randn(batch_size, 32)  # e.g., author age, report count, etc.

fusion_model = EarlyFusionEncoder()
with torch.no_grad():
    fused_output = fusion_model(text_embeddings, image_embeddings, metadata_features)

print(f"Text shape:     {text_embeddings.shape}")
print(f"Image shape:    {image_embeddings.shape}")
print(f"Metadata shape: {metadata_features.shape}")
print(f"Fused shape:    {fused_output.shape}")

<div style="border-left: 5px solid #f57c00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,152,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Model Development</span>
</div>

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Model selection</span>
</div>

A neural network is the most common model used for multi-task learning. In developing our model, we employ neural networks.

When choosing a neural network, what factors should be considered? It is necessary to determine the neural network's architectural design and the optimal hyperparameter selection, such as its hidden layers, activation function, learning rate, etc. The optimal choice of hyperparameters is usually determined by hyperparameter tuning. Let's briefly go over it.

Hyperparameter tuning is the process of finding the best values for hyperparameters in order to produce the best performance for a model. To tune hyperparameters, grid search is commonly used. The procedure involves training a new model for each combination of hyperparameter values, evaluating each model, then selecting the hyperparameters that lead to the best model. If you are interested in learning more about hyperparameter tuning, refer to [14].

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Model training</span>
</div>

<strong style="color: #ffad32;">Constructing the dataset</strong>

To train the multi-task classification model, we first need to construct the dataset. The dataset comprises model inputs (features) and outputs (labels) that the model is expected to predict. To construct inputs, we process posts offline in batches and compute fused features as described earlier. These features can be stored in a feature store for future training. In order to create labels for each input, we have two options:

* Hand labeling
* Natural labeling

With hand labeling, human contractors label posts manually. This option produces accurate labels, but it is expensive and time-consuming. With natural labeling, we rely on user reports to label posts automatically. While this option results in noisier labels, labels are produced more quickly. For the evaluation dataset, we use hand labeling to prioritize the accuracy of labels, and for the training dataset, we use natural labeling to prioritize labeling speed. A data point from the constructed dataset is shown in Figure 5.16.

![Figure 5.16: A constructed data point](assets/ch5-16-1-TXZGVSVN.png)

<strong style="color: #ffad32;">Choosing the loss function</strong>

Training a multi-task neural network is very similar to how we typically train neural network models. A forward propagation performs computations to make a prediction, a loss function measures the correctness of the prediction, and a backward propagation optimizes the model's parameters to reduce the loss in the next iteration. Let's examine the loss function. In multi-task training, each task is assigned a loss function based on its ML category. In our case, each task is framed as a binary classification, so we adopt a standard binary classification loss such as cross-entropy for each task. The overall loss is computed by combining task-specific losses, as shown in Figure 5.17.

![Figure 5.17: Model training](assets/ch5-17-1-TUCQRHJ4.png)

A common challenge in training multimodal systems is overfitting [15]. For example, when the learning speed varies across different modalities, one modality (e.g., image) can dominate the learning process. Two techniques to address this issue are gradient blending and focal loss. If you are interested in learning more about these techniques, refer to [16] [17].

<div style="border-left: 5px solid #f57c00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,152,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Evaluation</span>
</div>

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Offline metrics</span>
</div>

To evaluate the performance of a binary classification model, offline metrics such as precision, recall, and f1 score are commonly used. However, precision or recall alone is not sufficient to understand the overall performance. For example, a model with high precision might have a very low recall. The precision-recall (PR) curve and receiver operating characteristic (ROC) curve address those limitations. Let's explore each one.

<strong style="color: #ffad32;">PR-curve.</strong> PR curve shows the trade-off between precision and recall of the model. As Figure 5.18 shows, we obtain a PR curve by plotting the precision of the model using different probability thresholds, ranging from 0 to 1. To summarize the trade-offs between precision and recall, PR-AUC (the area under the precision-recall curve) calculates the area beneath the PR curve. In general, a high PR-AUC indicates a more accurate model.

![Figure 5.18: PR curve](assets/ch5-18-1-AM4GN6NU.png)

<strong style="color: #ffad32;">ROC curve.</strong> The ROC curve shows the trade-offs between the true positive rate (recall) and the false positive rate. Similar to the PR curve, ROC-AUC summarizes the model's performance by calculating the area under the ROC curve.

ROC and PR curves are two different ways to summarize the performance of a classification model. To learn about the differences between the PR curve and the ROC curve, read [18].

In our case, we use both ROC-AUC and PR-AUC as our offline metrics.

<div style="border-bottom: 2px solid rgba(255,152,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffad32;">Online metrics</span>
</div>

Let's explore a few important metrics to capture how safe the platform is.

<strong style="color: #ffad32;">Prevalence.</strong> This metric measures the ratio of harmful posts which we didn't prevent and all posts on the platform.

$$ \text{Prevalence} = \frac{\text{Number of harmful posts we didn't prevent}}{\text{Total number of posts on the platform}} $$

The shortcoming of this metric is that it treats harmful posts equally. For example, one harmful post with 100K views or impressions is more harmful than two posts with 10 views each.

<strong style="color: #ffad32;">Harmful impressions.</strong> We prefer this metric over prevalence. The reason is that the number of harmful posts on the platform does not show how many people were affected by those posts, whereas the number of harmful impressions does capture this information.

<strong style="color: #ffad32;">Valid appeals.</strong> Percentage of posts that were deemed harmful, but appealed and reversed.

$$ \text{Appeals} = \frac{\text{Number of reversed appeals}}{\text{Number of harmful posts detected by the system}} $$

<strong style="color: #ffad32;">Proactive rate.</strong> Percentage of harmful posts found and deleted by the system before users report it.

$$ \text{Proactive rate} = \frac{\text{Number of harmful posts detected by the system}}{\text{Number of harmful posts detected by the system} + \text{reported by users}} $$

<strong style="color: #ffad32;">User reports per harmful class.</strong> This metric measures the system's performance by looking into user reports for each harmful class.

In [ ]:
# ── Code Augmentation: PR-AUC & ROC-AUC Evaluation ──

import numpy as np
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    auc,
    average_precision_score,
    roc_auc_score
)
import matplotlib.pyplot as plt

# Simulate ground truth labels and predicted probabilities for the 'violence' task
np.random.seed(42)
y_true = np.random.randint(0, 2, size=500)
# Generate scores that are somewhat correlated with truth
y_scores = y_true * np.random.uniform(0.5, 1.0, size=500) + \
           (1 - y_true) * np.random.uniform(0.0, 0.5, size=500)

# ── Compute PR-AUC ──
precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc = average_precision_score(y_true, y_scores)

# ── Compute ROC-AUC ──
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc_val = roc_auc_score(y_true, y_scores)

# ── Visualize ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR Curve
axes[0].plot(recall, precision, color='#2196F3', lw=2)
axes[0].fill_between(recall, precision, alpha=0.15, color='#2196F3')
axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title(f'Precision-Recall Curve (PR-AUC = {pr_auc:.3f})', fontsize=13)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])
axes[0].grid(alpha=0.3)

# ROC Curve
axes[1].plot(fpr, tpr, color='#4CAF50', lw=2)
axes[1].fill_between(fpr, tpr, alpha=0.15, color='#4CAF50')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[1].set_title(f'ROC Curve (ROC-AUC = {roc_auc_val:.3f})', fontsize=13)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

<div style="border-left: 5px solid #f57c00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,152,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Serving</span>
</div>

Figure 5.19 shows the high-level ML system design. Let's take a closer look at each component.

![Figure 5.19: ML system design](assets/ch5-19-1-AXVUQWD5.png)

<strong style="color: #ffad32;">Harmful content detection service</strong>

Given a new post, this service predicts the probability of harm. According to the requirements, some types of harm should be handled immediately due to their sensitivity. When this happens, the violation enforcement service removes the post immediately.

<strong style="color: #ffad32;">Violation enforcement service</strong>

The violation enforcement service immediately takes down a post if the harmful content detection service predicts harm with high confidence. It also notifies the user why the post was removed.

<strong style="color: #ffad32;">Demoting service</strong>

If the harmful content detection service predicts harm with low confidence, the demoting service temporarily demotes the post in order to decrease the chance of it spreading among users.

Then, the post is stored in storage for manual review by humans. The review team manually reviews the post and assigns a label from one of the predefined classes of harm. We will use these labeled posts in future training iterations to improve the model.

<div style="border-left: 5px solid #f57c00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,152,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Other Talking Points</span>
</div>

* Handle biases introduced by human labeling [19].
* Adapt the system to detect trending harmful classes (e.g., Covid-19, elections) [20].
* How to build a harmful content detection system that leverages temporal information such as users' sequence of actions [21][22].
* How to effectively select post samples for human review [23].
* How to detect authentic and fake accounts [24].
* How to deal with borderline contents [25], i.e., types of content that are not prohibited by guidelines, but come close to the red lines drawn by those policies.
* How to make the harmful content detection system efficient, so we can deploy it on-device [26].
* How to substitute Transformer-based architectures with linear Transformers to create a more efficient system [27] [28].

<div style="border-left: 5px solid #f57c00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,152,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">References</span>
</div>

1. Facebook's inauthentic behavior. https://transparency.fb.com/policies/community-standards/inauthentic-behavior/.
2. LinkedIn's professional community policies. https://www.linkedin.com/legal/professional-community-policies.
3. Twitter's civic integrity policy. https://help.twitter.com/en/rules-and-policies/election-integrity-policy.
4. Facebook's integrity survey. https://arxiv.org/pdf/2009.10311.pdf.
5. Pinterest's violation detection system. https://medium.com/pinterest-engineering/how-pinterest-fights-misinformation-hate-speech-and-self-harm-content-with-machine-learning-1806b73b40ef.
6. Abusive detection at LinkedIn. https://engineering.linkedin.com/blog/2019/isolation-forest.
7. WPIE method. https://ai.facebook.com/blog/community-standards-report/.
8. BERT paper. https://arxiv.org/pdf/1810.04805.pdf.
9. Multilingual DistilBERT. https://huggingface.co/distilbert-base-multilingual-cased.
10. Multilingual language models. https://arxiv.org/pdf/2107.00676.pdf.
11. CLIP model. https://openai.com/blog/clip/.
12. SimCLR paper. https://arxiv.org/pdf/2002.05709.pdf.
13. VideoMoCo paper. https://arxiv.org/pdf/2103.05905.pdf.
14. Hyperparameter tuning. https://cloud.google.com/ai-platform/training/docs/hyperparameter-tuning-overview.
15. Overfitting. https://en.wikipedia.org/wiki/Overfitting.
16. Focal loss. https://amaarora.github.io/posts/2020-06-29-FocalLoss.html.
17. Gradient blending in multimodal systems. https://arxiv.org/pdf/1905.12681.pdf.
18. ROC curve vs precision-recall curve. https://machinelearningmastery.com/roc-curves-and-precision-recall-curves-for-classification-in-python/.
19. Introduced bias by human labeling. https://labelyourdata.com/articles/bias-in-machine-learning.
20. Facebook's approach to quickly tackling trending harmful content. https://ai.facebook.com/blog/harmful-content-can-evolve-quickly-our-new-ai-system-adapts-to-tackle-it/.
21. Facebook's TIES approach. https://arxiv.org/pdf/2002.07917.pdf.
22. Temporal interaction embedding. https://www.facebook.com/atscaleevents/videos/730968530723238/.
23. Building and scaling human review system. https://www.facebook.com/atscaleevents/videos/1201751883328695/.
24. Abusive account detection framework. https://www.youtube.com/watch?v=YeX4MdU0JNk.
25. Borderline contents. https://transparency.fb.com/features/approach-to-ranking/content-distribution-guidelines/content-borderline-to-the-community-standards/.
26. Efficient harmful content detection. https://about.fb.com/news/2021/12/metas-new-ai-system-tackles-harmful-content/.
27. Linear Transformer paper. https://arxiv.org/pdf/2006.04768.pdf.
28. Efficient AI models to detect hate speech. https://ai.facebook.com/blog/how-facebook-uses-super-efficient-ai-models-to-detect-hate-speech/.